In [0]:
# 03_gold_features.py
# Notebook: 03_gold_features
# Ejecutar en Databricks (Python)

from pyspark.sql import SparkSession, functions as F, Window
from functools import reduce
spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# AJUSTE DE RUTAS A UNITY CATALOG VOLUMES
# *******************************************************************
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
SILVER_VOLUME_NAME = "silver_data"
GOLD_VOLUME_NAME = "gold_data"

# Rutas de Origen y Destino
silver_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{SILVER_VOLUME_NAME}/"
gold_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{GOLD_VOLUME_NAME}/"

# Formato de Fecha/Timestamp
DATE_FORMAT = 'yyyy-MM-dd HH:mm:ss'
# Se usa el literal de Spark para evitar el error UNRESOLVED_COLUMN
DATE_FORMAT_LITERAL = F.lit(DATE_FORMAT) 

# *******************************************************************
# PASO 1: Crear el Volume de destino (Gold) si no existe
# *******************************************************************
try:
    print(f"Verificando y creando el Volume de destino Gold: {CATALOG_NAME}.{SCHEMA_NAME}.{GOLD_VOLUME_NAME}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{GOLD_VOLUME_NAME}")
    print("Volume Gold verificado/creado exitosamente.")
except Exception as e:
    print(f"ERROR: No se pudo crear el Volume Gold. Verifica tus permisos de Unity Catalog.")
    print(f"Detalle del error de creación: {e}")
    raise # Detener la ejecución si el destino no se puede asegurar


# *******************************************************************
# PASO 2: Cargar tablas Silver
# *******************************************************************
print("Cargando tablas de la capa Silver...")
customers = spark.read.parquet(silver_path + "customers")
orders = spark.read.parquet(silver_path + "orders_full")
order_items = spark.read.parquet(silver_path + "order_items")
products = spark.read.parquet(silver_path + "products")
# NOTA: También podrías necesitar 'translation' aquí si quieres usar nombres de categorías en español.


# *******************************************************************
# PASO 3: Feature Engineering
# *******************************************************************
print("Generando Features...")

# Asegurarnos que orders tiene customer_id y columnas útiles
# Se usa to_date con timestamp para convertir la columna existente de timestamp
orders = orders.withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))
orders = orders.withColumn("delivered_lag_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))

# ----------------- AGGREGATIONS: actividad y valor -----------------
agg_activity = orders.groupBy("customer_id").agg(
    F.countDistinct("order_id").alias("num_orders"),
    F.count(F.when(F.col("order_status") == "delivered", True)).alias("num_orders_delivered"),
    F.count(F.when(F.col("order_status") == "canceled", True)).alias("num_orders_canceled"),
    F.min("order_purchase_timestamp").alias("first_purchase_ts"),
    F.max("order_purchase_timestamp").alias("last_purchase_ts"),
    F.avg("order_sum_price").alias("avg_order_value"),
    F.stddev("order_sum_price").alias("std_order_value"),
    F.max("order_sum_price").alias("max_order_value"),
    F.sum("order_sum_price").alias("total_spent"),
    F.avg("items_count").alias("avg_items_per_order")
)

# ----------------- RECENCIA / FREQ DERIVADAS -----------------
# recency en días (reference date: la fecha de ejecución)
today = F.current_date()
agg_activity = agg_activity.withColumn("recency_days", F.datediff(today, F.to_date("last_purchase_ts")))

# orders per month estimate
orders_months = orders.withColumn("order_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
orders_months_agg = orders_months.groupBy("customer_id").agg(
    F.countDistinct("order_month").alias("distinct_months"),
    F.min("order_month").alias("first_month"),
    F.max("order_month").alias("last_month")
)
agg_activity = agg_activity.join(orders_months_agg, on="customer_id", how="left")
agg_activity = agg_activity.withColumn("orders_per_month_est",
                                         F.col("num_orders") / F.greatest(F.lit(1), F.col("distinct_months")))

# ----------------- FEATURES POR CATEGORIA / PRODUCT -----------------
# join order_items para features relacionadas a products
prod_by_customer = order_items.join(orders.select("order_id","customer_id"), on="order_id", how="left") \
    .groupBy("customer_id").agg(
        F.countDistinct("product_id").alias("distinct_products_total"),
        F.count("product_id").alias("total_items_bought"),
        F.avg("price").alias("avg_product_price"),
        F.max("price").alias("max_product_price"),
        F.sum("price").alias("sum_price_items")
    )

# categoría más frecuente (si products tiene category_name)
order_prod = order_items.join(products.select("product_id","product_category_name"), on="product_id", how="left") \
                         .join(orders.select("order_id","customer_id"), on="order_id", how="left")

cat_freq = order_prod.groupBy("customer_id", "product_category_name").agg(F.count("*").alias("cnt")) \
    .withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("cnt")))) \
    .filter(F.col("rn")==1).select("customer_id", F.col("product_category_name").alias("top_category"))

# ----------------- REVIEWS FEATURES -----------------
reviews = orders.select("customer_id","review_score","review_creation_date").where(F.col("review_score").isNotNull())
reviews_agg = reviews.groupBy("customer_id").agg(
    F.avg("review_score").alias("avg_review_score"),
    F.count("review_score").alias("num_reviews"),
    F.min("review_creation_date").alias("first_review_date"),
    F.max("review_creation_date").alias("last_review_date")
)

# ----------------- PAYMENT FEATURES (CORREGIDO) -----------------
# Se eliminó la selección de columnas inexistentes (payment_type, etc.)
# y se usa directamente las agregaciones de orden disponibles en el DF 'orders'
payments_agg = orders.groupBy("customer_id").agg(
    # El número máximo de tipos de pago distintos utilizados en CUALQUIER orden de ese cliente
    F.max("n_payment_types").alias("max_payment_types_per_order"),
    # El valor promedio de pago por orden (usando la suma pre-agregada)
    F.avg("payment_sum").alias("avg_order_payment_sum"), 
    # El promedio de cuotas (installments) en todas sus órdenes
    F.avg("avg_installments").alias("avg_installments_across_orders")
)

# ----------------- LOGISTICS -----------------
logistic_agg = orders.groupBy("customer_id").agg(
    F.avg("delivered_lag_days").alias("avg_delivery_days"),
    F.count(F.when(F.col("delivered_lag_days") < 3, True)).alias("fast_deliveries"),
    F.count(F.when(F.col("delivered_lag_days") > 15, True)).alias("slow_deliveries")
)

# ----------------- JOIN TODO -----------------
dfs = [agg_activity, prod_by_customer, cat_freq, reviews_agg, payments_agg, logistic_agg]
features = reduce(lambda left, right: left.join(right, on="customer_id", how="left"), dfs)

# ----------------- FEATURE ENGINEERING ADICIONAL (crear más columnas para llegar a ~70) -----------------
# Ejemplos: ratios, flags, interaction terms, percentiles
features = features.withColumn("spend_per_item", F.when(F.col("distinct_products_total").isNotNull(),
                                                     F.col("total_spent") / F.col("distinct_products_total")).otherwise(None))
features = features.withColumn("high_value_flag", F.when(F.col("avg_order_value") > 200, 1).otherwise(0))
features = features.withColumn("loyal_customer_flag", F.when(F.col("num_orders") >= 5, 1).otherwise(0))
features = features.withColumn("avg_order_value_times_freq", F.col("avg_order_value") * F.col("orders_per_month_est"))

# Generar percentiles del ticket por cliente (ejemplo con approx quantile por cliente no trivial en spark; se puede aproximar por global)
# Aquí añadimos columnas auxiliares (duplicar patrones para obtener más features)
features = features.withColumn("avg_order_value_sq", F.col("avg_order_value") * F.col("avg_order_value"))
features = features.withColumn("order_value_cv", F.when(F.col("avg_order_value")!=0, F.col("std_order_value")/F.col("avg_order_value")).otherwise(None))

# Puedes añadir más flags y combinaciones para alcanzar 70 columnas. Asegúrate de documentarlas.
# ----------------- GUARDAR GOLD -----------------
features.write.mode("overwrite").parquet(gold_path + "customer_features")

# Mostrar resumen
print("Features guardadas en:", gold_path + "customer_features")
print("Número aproximado de columnas:", len(features.columns))
display(features.limit(5))